In [ ]:
# 03_agente.ipynb

import anthropic
import os
import time
import json
from dotenv import load_dotenv

load_dotenv()
client_anthropic = anthropic.Anthropic()


# Las mismas 3 empresas que usaste en semana 2
# Asegúrate de que la carpeta corpus/ sigue ahí

# Definimos las herramientas que el agente puede usar
tools = [
    {
        "name": "list_files",
        "description": "Lista todos los ficheros disponibles en el corpus con su nombre. Útil para saber qué documentos existen antes de leerlos.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "read_file",
        "description": "Lee el contenido completo de un fichero de texto.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Nombre del fichero a leer, tal como aparece en list_files"
                }
            },
            "required": ["filename"]
        }
    },
    {
        "name": "search_in_file",
        "description": "Busca un término o frase dentro de un fichero y devuelve las líneas que lo contienen con contexto.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "Nombre del fichero donde buscar"
                },
                "query": {
                    "type": "string",
                    "description": "Término o frase a buscar"
                }
            },
            "required": ["filename", "query"]
        }
    }
]

# Implementación real de cada herramienta
def list_files():
    files = os.listdir("corpus")
    return {"files": files, "count": len(files)}

def read_file(filename):
    path = f"corpus/{filename}"
    if not os.path.exists(path):
        return {"error": f"Fichero '{filename}' no encontrado"}
    with open(path, encoding="utf-8") as f:
        content = f.read()
    return {"filename": filename, "content": content, "chars": len(content)}

def search_in_file(filename, query):
    path = f"corpus/{filename}"
    if not os.path.exists(path):
        return {"error": f"Fichero '{filename}' no encontrado"}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    
    query_lower = query.lower()
    results = []
    for i, line in enumerate(lines):
        if query_lower in line.lower():
            # Devuelve la línea con 2 líneas de contexto arriba y abajo
            start = max(0, i - 2)
            end = min(len(lines), i + 3)
            context = "".join(lines[start:end]).strip()
            results.append({"line": i+1, "context": context})
    
    return {"query": query, "matches": results, "total": len(results)}

def execute_tool(tool_name, tool_input):
    if tool_name == "list_files":
        return list_files()
    elif tool_name == "read_file":
        return read_file(tool_input["filename"])
    elif tool_name == "search_in_file":
        return search_in_file(tool_input["filename"], tool_input["query"])
    else:
        return {"error": f"Herramienta desconocida: {tool_name}"}

print("Herramientas definidas correctamente")
print(f"Ficheros disponibles: {list_files()}")

In [ ]:
def agent_query(pregunta, verbose=True):
    messages = [{"role": "user", "content": pregunta}]
    tool_calls_count = 0
    start_time = time.time()
    
    if verbose:
        print(f"Pregunta: {pregunta}")
        print("-" * 50)
    
    while True:
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1000,
            tools=tools,
            messages=messages
        )
        
        # Si el modelo quiere usar una herramienta
        if response.stop_reason == "tool_use":
            # Añadir respuesta del modelo al historial
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_calls_count += 1
                    tool_name = block.name
                    tool_input = block.input
                    
                    if verbose:
                        print(f"  Herramienta [{tool_calls_count}]: {tool_name}({tool_input})")
                    
                    result = execute_tool(tool_name, tool_input)
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result, ensure_ascii=False)
                    })
            
            messages.append({"role": "user", "content": tool_results})
        
        # Si el modelo ya tiene la respuesta final
        elif response.stop_reason == "end_turn":
            elapsed = time.time() - start_time
            final_answer = next(
                (b.text for b in response.content if hasattr(b, "text")), ""
            )
            
            if verbose:
                print(f"\nRespuesta final:\n{final_answer}")
                print(f"\nResumen: {tool_calls_count} llamadas a herramientas, {elapsed:.2f}s")
            
            return {
                "answer": final_answer,
                "tool_calls": tool_calls_count,
                "latency": elapsed
            }

# Prueba básica
agent_query("¿Cuál fue el revenue de empresa Alpha en 2024?")

Pregunta: ¿Cuál fue el revenue de empresa Alpha en 2024?
--------------------------------------------------


/tmp/ipykernel_42099/1195034023.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  Herramienta [1]: list_files({})
  Herramienta [2]: read_file({'filename': 'empresa_alpha_2024.txt'})

Respuesta final:
Según el informe anual de la Empresa Alpha para 2024, el **revenue total fue de 47.3 millones de euros**.

Detalles adicionales sobre el revenue de la empresa Alpha en 2024:

- **Revenue total**: 47.3 millones de euros
- **Crecimiento**: 18.4% respecto a 2023 (cuando fue de 39.9 millones)
- **Desglose por segmentos**:
  - Segmento de software: 32.2 millones de euros (68% del total)
  - Servicios profesionales: 15.1 millones de euros (32% del total)
- **Desglose geográfico**:
  - España: 21.3 millones de euros (45%)
  - Europa central: 16.6 millones de euros (35%)
  - Latinoamérica: 9.5 millones de euros (20%)

Resumen: 2 llamadas a herramientas, 10.00s


{'answer': 'Según el informe anual de la Empresa Alpha para 2024, el **revenue total fue de 47.3 millones de euros**.\n\nDetalles adicionales sobre el revenue de la empresa Alpha en 2024:\n\n- **Revenue total**: 47.3 millones de euros\n- **Crecimiento**: 18.4% respecto a 2023 (cuando fue de 39.9 millones)\n- **Desglose por segmentos**:\n  - Segmento de software: 32.2 millones de euros (68% del total)\n  - Servicios profesionales: 15.1 millones de euros (32% del total)\n- **Desglose geográfico**:\n  - España: 21.3 millones de euros (45%)\n  - Europa central: 16.6 millones de euros (35%)\n  - Latinoamérica: 9.5 millones de euros (20%)',
 'tool_calls': 2,
 'latency': 9.996628522872925}

In [4]:
# FALLO 1 del RAG: pregunta que cruza información de dos chunks
print("=" * 60)
print("CASO 1: Pregunta que cruza documentos")
print("=" * 60)
r1 = agent_query("¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?")

CASO 1: Pregunta que cruza documentos
Pregunta: ¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?
--------------------------------------------------


/tmp/ipykernel_42099/1195034023.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  Herramienta [1]: list_files({})
  Herramienta [2]: search_in_file({'filename': 'empresa_alpha_2024.txt', 'query': 'LTV/CAC'})
  Herramienta [3]: search_in_file({'filename': 'empresa_alpha_2024.txt', 'query': 'EBITDA'})
  Herramienta [4]: search_in_file({'filename': 'empresa_beta_2024.txt', 'query': 'LTV/CAC'})
  Herramienta [5]: search_in_file({'filename': 'empresa_beta_2024.txt', 'query': 'EBITDA'})
  Herramienta [6]: search_in_file({'filename': 'empresa_gamma_2024.txt', 'query': 'LTV/CAC'})
  Herramienta [7]: search_in_file({'filename': 'empresa_gamma_2024.txt', 'query': 'EBITDA'})
  Herramienta [8]: search_in_file({'filename': 'empresa_alpha_2024.txt', 'query': 'CAC'})
  Herramienta [9]: search_in_file({'filename': 'empresa_alpha_2024.txt', 'query': 'LTV'})
  Herramienta [10]: search_in_file({'filename': 'empresa_beta_2024.txt', 'query': 'CAC'})
  Herramienta [11]: search_in_file({'filename': 'empresa_beta_2024.txt', 'query': 'LTV'})
  Herramienta [12]: read_file({'filename': 'emp

In [5]:
# FALLO 2 del RAG: comparativa entre documentos
print("=" * 60)
print("CASO 2: Comparativa entre empresas")
print("=" * 60)
r2 = agent_query("¿Qué empresa creció más en revenue durante 2024?")

CASO 2: Comparativa entre empresas
Pregunta: ¿Qué empresa creció más en revenue durante 2024?
--------------------------------------------------


/tmp/ipykernel_42099/1195034023.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  Herramienta [1]: list_files({})
  Herramienta [2]: read_file({'filename': 'empresa_alpha_2024.txt'})
  Herramienta [3]: read_file({'filename': 'empresa_beta_2024.txt'})
  Herramienta [4]: read_file({'filename': 'empresa_gamma_2024.txt'})

Respuesta final:
Basándome en la información de los informes anuales de las tres empresas, aquí está el análisis del crecimiento de revenue durante 2024:

## Crecimiento de Revenue en 2024:

**Empresa Gamma: 67% de crecimiento**
- Revenue 2024: 8.9 millones de euros
- Revenue 2023: 5.3 millones de euros
- Crecimiento: 67%

**Empresa Alpha: 18.4% de crecimiento**  
- Revenue 2024: 47.3 millones de euros
- Revenue 2023: 39.9 millones de euros
- Crecimiento: 18.4%

**Empresa Beta: 6.2% de crecimiento**
- Revenue 2024: 124.7 millones de euros  
- Revenue 2023: 117.4 millones de euros
- Crecimiento: 6.2%

## Conclusión:

**Empresa Gamma creció más en revenue durante 2024** con un crecimiento del 67%. Aunque es la empresa más pequeña en términos de volume

In [10]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.Client()
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)
chroma_client.delete_collection("informes_v2")
collection = chroma_client.create_collection(name="informes_v2", embedding_function=ef)

import os
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+chunk_size]))
        i += chunk_size - overlap
    return chunks

for filename in os.listdir("corpus"):
    with open(f"corpus/{filename}", encoding="utf-8") as f:
        text = f.read()
    chunks = chunk_text(text)
    for j, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            ids=[f"{filename}_chunk_{j}"],
            metadatas=[{"source": filename}]
        )

def rag_query_timed(pregunta, n_results=3):
    start = time.time()
    results = collection.query(query_texts=[pregunta], n_results=n_results)
    chunks_recuperados = results["documents"][0]
    contexto = "\n\n---\n\n".join(chunks_recuperados)
    prompt = f"""Basándote únicamente en los siguientes fragmentos, responde la pregunta.
Si la información no está en los fragmentos, dilo explícitamente.

FRAGMENTOS:
{contexto}

PREGUNTA: {pregunta}
RESPUESTA:"""
    response = client_anthropic.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    elapsed = time.time() - start
    return {"answer": response.content[0].text, "latency": round(elapsed, 2)}

In [11]:
casos = [
    "¿Cuál fue el revenue de empresa Alpha en 2024?",
    "¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?",
    "¿Qué empresa creció más en revenue durante 2024?",
    "¿Cuál es exactamente el beneficio neto de Gamma en 2024?"
]

rag_latencia      = []
rag_respuestas    = []
agente_latencia   = []
agente_tool_calls = []
agente_respuestas = []

for i, pregunta in enumerate(casos):
    print(f"\n{'='*60}")
    print(f"CASO {i+1}: {pregunta}")
    print(f"{'='*60}")

    print("\n--- RAG ---")
    r_rag = rag_query_timed(pregunta)
    rag_latencia.append(r_rag["latency"])
    rag_respuestas.append(r_rag["answer"])
    print(f"Respuesta: {r_rag['answer'][:200]}")
    print(f"Latencia: {r_rag['latency']}s")

    print("\n--- AGENTE ---")
    r_agente = agent_query(pregunta, verbose=True)
    agente_latencia.append(r_agente["latency"])
    agente_tool_calls.append(r_agente["tool_calls"])
    agente_respuestas.append(r_agente["answer"])

print("\n\n" + "="*60)
print("INTRODUCE MANUALMENTE SI CADA RESPUESTA FUE CORRECTA")
print("="*60)
for i, caso in enumerate(casos):
    print(f"\nCaso {i+1}: {caso}")
    print(f"  RAG:    {rag_respuestas[i][:150]}")
    print(f"  Agente: {agente_respuestas[i][:150]}")


CASO 1: ¿Cuál fue el revenue de empresa Alpha en 2024?

--- RAG ---


/tmp/ipykernel_42099/141567045.py:45: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


Respuesta: Según los fragmentos proporcionados, el revenue de la empresa Alpha en 2024 fue de 47.3 millones de euros.
Latencia: 1.8s

--- AGENTE ---
Pregunta: ¿Cuál fue el revenue de empresa Alpha en 2024?
--------------------------------------------------


/tmp/ipykernel_42099/1195034023.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  Herramienta [1]: list_files({})
  Herramienta [2]: read_file({'filename': 'empresa_alpha_2024.txt'})

Respuesta final:
Según el informe anual de la empresa Alpha de 2024, el revenue total del ejercicio fue de **47.3 millones de euros**.

Además de esta información principal, aquí tienes algunos datos adicionales relevantes:

- Esto representó un crecimiento del 18.4% respecto al año anterior (39.9 millones de euros en 2023)
- El revenue se distribuyó en dos segmentos:
  - Software: 32.2 millones de euros (68% del total)
  - Servicios profesionales: 15.1 millones de euros (32% del total)
- Por mercados geográficos:
  - España: 21.3 millones de euros (45%)
  - Europa central: 16.6 millones de euros (35%)
  - Latinoamérica: 9.5 millones de euros (20%)

Resumen: 2 llamadas a herramientas, 7.98s

CASO 2: ¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?

--- RAG ---
Respuesta: Basándome en los fragmentos proporcionados:

**Empresa con mejor ratio LTV/CAC:** Gamma es la ún

<div style="font-size:14">

# Nota sobre herramientas

Hemos definido herramientas (`list_files`, `read_file`, `search_in_file`) para ver claramente qué hace el agente paso a paso.

En producción sería más simple dar acceso directo a **bash** y usar `grep` y `ls` sin intermediarios.

---

## 🧭 Evolución

Añadimos **wikilinks** al corpus: `[[nombre-fichero]]`.

Así los documentos se conectan entre sí como en **Obsidian**, y el agente navega el grafo de conocimiento en lugar de buscar a ciegas.

</div>

In [13]:
# ============================================================
# BLOQUE 6: Agente con wikilinks — navegación por grafo
# ============================================================

# Primero creamos una mini vault con wikilinks entre documentos
os.makedirs("vault", exist_ok=True)

vault = {
    "00-index.md": """
# Index

Punto de entrada de la vault. Aquí están todas las empresas analizadas.

## Empresas
- [[empresa-alpha]] — Software B2B, alto crecimiento
- [[empresa-beta]] — Manufactura industrial, márgenes presionados  
- [[empresa-gamma]] — SaaS de gestión de flota, fase inversión

## Análisis transversal
- [[comparativa-financiera]]
- [[riesgos-sector]]
""",

    "empresa-alpha.md": """
# Empresa Alpha

## Resultados 2024
Revenue: 47.3M euros (+18.4% vs 2023)
EBITDA: 12.1M euros, margen 25.6%
Beneficio neto: 6.8M euros
Deuda neta: 3.2M euros (desde 8.1M en 2023)

## Segmentos
- Software: 68% del revenue (32.2M), crecimiento +24%
- Servicios profesionales: 32% (15.1M)

## Geografía
- España: 45% (21.3M)
- Europa central: 35% (16.6M)
- Latinoamérica: 20% (9.5M)

## Perspectivas 2025
Crecimiento esperado: 15-20%
Inversión I+D: 4.2M euros
Objetivo EBITDA margin: >27%

## Ver también
- [[comparativa-financiera]] para contexto vs competidores
- [[riesgos-sector]] para riesgos específicos de software B2B
""",

    "empresa-beta.md": """
# Empresa Beta

## Resultados 2024
Revenue: 124.7M euros (+6.2% vs 2023)
EBITDA: 18.3M euros, margen 14.7%
Beneficio neto: 2.1M euros (desde 5.4M en 2023, caída fuerte)
Deuda neta: 31.4M euros (aumentó)

## Segmentos
- Manufactura industrial: 78.6M (63%)
- Distribución: 46.1M (37%)

## Geografía
- Mercado doméstico (España/Portugal): 89%
- Exportación (Francia, Italia): 11%

## Perspectivas 2025
Entorno desafiante por inflación de costes energéticos
Plan de reestructuración: reducción de plantilla del 8%
Objetivo revenue: 120-128M euros

## Ver también
- [[comparativa-financiera]] para contexto vs Alpha y Gamma
- [[riesgos-sector]] para riesgos de manufactura y energía
""",

    "empresa-gamma.md": """
# Empresa Gamma

## Resultados 2024
Revenue: 8.9M euros (+67% vs 2023)
EBITDA: -2.4M euros (fase inversión)
Beneficio neto: -3.1M euros
Caja disponible: 14.2M euros (tras Serie B)

## Métricas SaaS
ARR: 7.8M euros (+71% interanual)
NRR: 118% — expansión en clientes existentes
Churn rate: 4.2% anual
Clientes activos: 342 empresas

## Unit economics
CAC: 18.400 euros
LTV: 94.000 euros
Ratio LTV/CAC: 5.1x

## Perspectivas 2025
Objetivo ARR: 13M euros
Expansión Francia y Alemania en Q2
EBITDA breakeven previsto en Q4 2025

## Ver también
- [[comparativa-financiera]] para contexto vs empresas rentables
- [[riesgos-sector]] para riesgos de expansión internacional
""",

    "comparativa-financiera.md": """
# Comparativa Financiera 2024

Este documento consolida los datos de [[empresa-alpha]], [[empresa-beta]]
y [[empresa-gamma]] para análisis comparativo.

## Revenue y crecimiento
| Empresa | Revenue | Crecimiento |
|---------|---------|-------------|
| Alpha   | 47.3M   | +18.4%      |
| Beta    | 124.7M  | +6.2%       |
| Gamma   | 8.9M    | +67%        |

Gamma lidera en crecimiento relativo pero parte de base pequeña.
Alpha tiene el mejor equilibrio entre escala y crecimiento.
Beta es la más grande pero crece poco y pierde margen.

## Rentabilidad
| Empresa | EBITDA margin | Beneficio neto |
|---------|--------------|----------------|
| Alpha   | 25.6%        | 6.8M           |
| Beta    | 14.7%        | 2.1M           |
| Gamma   | Negativo     | -3.1M          |

Alpha es claramente la más rentable en términos relativos.

## Deuda y solidez
- Alpha: deuda neta reducida a 3.2M — posición saneada
- Beta: deuda neta 31.4M y creciendo — riesgo financiero
- Gamma: caja 14.2M, sin deuda — runway suficiente para plan 2025
""",

    "riesgos-sector.md": """
# Riesgos por Sector

## Software B2B (ver [[empresa-alpha]])
- Ciclos de venta largos si macro se deteriora
- Dependencia de renovaciones en segmento enterprise
- Presión competitiva de players globales en Europa central

## Manufactura industrial (ver [[empresa-beta]])
- Exposición directa a costes energéticos
- Concentración geográfica elevada (89% doméstico)
- Reestructuración en curso añade riesgo de ejecución

## SaaS / Expansion stage (ver [[empresa-gamma]])
- Burn rate activo, dependiente de siguiente ronda si no llega breakeven
- Expansión internacional simultánea en dos mercados es ambiciosa
- NRR del 118% es positivo pero churn del 4.2% hay que monitorizar
"""
}

for filename, content in vault.items():
    with open(f"vault/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("Vault creada:")
for f in os.listdir("vault"):
    print(f"  {f}")

Vault creada:
  00-index.md
  comparativa-financiera.md
  empresa-alpha.md
  empresa-beta.md
  empresa-gamma.md
  riesgos-sector.md


In [14]:
import re

# Herramienta nueva: parsear y seguir wikilinks
def extract_wikilinks(content):
    """Extrae todos los [[wikilinks]] de un texto."""
    return re.findall(r'\[\[([^\]]+)\]\]', content)

def follow_link(link_name):
    """Lee el fichero referenciado por un wikilink."""
    filename = f"{link_name}.md"
    path = f"vault/{filename}"
    if not os.path.exists(path):
        return {"error": f"Documento '{filename}' no encontrado en la vault"}
    with open(path, encoding="utf-8") as f:
        content = f.read()
    links = extract_wikilinks(content)
    return {
        "filename": filename,
        "content": content,
        "wikilinks_found": links
    }

# Herramientas del agente vault — igual que antes pero sobre /vault
vault_tools = [
    {
        "name": "list_vault",
        "description": "Lista todos los documentos disponibles en la vault.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "read_document",
        "description": "Lee un documento de la vault por nombre de fichero.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "Nombre del fichero incluyendo .md"}
            },
            "required": ["filename"]
        }
    },
    {
        "name": "follow_wikilink",
        "description": "Sigue un wikilink [[nombre]] y lee el documento referenciado. Úsalo cuando encuentres referencias [[entre corchetes]] en un documento y necesites leer ese contenido.",
        "input_schema": {
            "type": "object",
            "properties": {
                "link_name": {"type": "string", "description": "El nombre dentro de los corchetes, sin [[ ]]"}
            },
            "required": ["link_name"]
        }
    }
]

def execute_vault_tool(tool_name, tool_input):
    if tool_name == "list_vault":
        files = os.listdir("vault")
        return {"files": files, "count": len(files)}
    elif tool_name == "read_document":
        filename = tool_input["filename"]
        path = f"vault/{filename}"
        if not os.path.exists(path):
            return {"error": f"'{filename}' no encontrado"}
        with open(path, encoding="utf-8") as f:
            content = f.read()
        links = extract_wikilinks(content)
        return {"filename": filename, "content": content, "wikilinks_found": links}
    elif tool_name == "follow_wikilink":
        return follow_link(tool_input["link_name"])
    else:
        return {"error": f"Herramienta desconocida: {tool_name}"}

print("Herramientas de vault definidas")

Herramientas de vault definidas


In [ ]:
#search_in_file tiene sentido añadirlo cuando los documentos son largos, por ejemplo un informe anual real de 50 páginas en texto plano.
# En ese caso leer el fichero entero cada vez es costoso en tokens y latencia,
# y tiene más sentido buscar primero el término concreto y leer solo ese fragmento con contexto.

In [ ]:
def vault_agent_query(pregunta, verbose=True):
    messages = [{"role": "user", "content": pregunta}]
    tool_calls_count = 0
    start_time = time.time()

    if verbose:
        print(f"Pregunta: {pregunta}")
        print("-" * 50)

    while True:
        response = client_anthropic.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1500,
            tools=vault_tools,
            system="Tienes acceso a una vault de documentos conectados por wikilinks [[nombre]]. "
                   "Empieza siempre por 00-index.md para orientarte. "
                   "Cuando encuentres un [[wikilink]] relevante para responder la pregunta, síguelo con follow_wikilink.",
            messages=messages
        )

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_calls_count += 1
                    if verbose:
                        print(f"  [{tool_calls_count}] {block.name}({block.input})")
                    result = execute_vault_tool(block.name, block.input)
                    # Muestra los wikilinks encontrados para que veas la navegación
                    if verbose and "wikilinks_found" in result:
                        print(f"       → wikilinks encontrados: {result['wikilinks_found']}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result, ensure_ascii=False)
                    })
            messages.append({"role": "user", "content": tool_results})

        elif response.stop_reason == "end_turn":
            elapsed = time.time() - start_time
            final_answer = next(
                (b.text for b in response.content if hasattr(b, "text")), ""
            )
            if verbose:
                print(f"\nRespuesta:\n{final_answer}")
                print(f"\nResumen: {tool_calls_count} tool calls — {elapsed:.2f}s")
            return {"answer": final_answer, "tool_calls": tool_calls_count, "latency": elapsed}

In [16]:
# Prueba 1: pregunta simple — ¿sigue el índice o va directo?
vault_agent_query("¿Cuál fue el revenue de empresa Alpha en 2024?")

Pregunta: ¿Cuál fue el revenue de empresa Alpha en 2024?
--------------------------------------------------


/tmp/ipykernel_42099/506747412.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  [1] read_document({'filename': '00-index.md'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma', 'comparativa-financiera', 'riesgos-sector']
  [2] follow_wikilink({'link_name': 'empresa-alpha'})
       → wikilinks encontrados: ['comparativa-financiera', 'riesgos-sector']

Respuesta:
Según la información de la vault, **el revenue de la empresa Alpha en 2024 fue de 47.3 millones de euros**, lo que representa un crecimiento del 18.4% comparado con 2023.

Detalles adicionales sobre sus resultados financieros en 2024:
- **EBITDA**: 12.1M euros (margen del 25.6%)
- **Beneficio neto**: 6.8M euros
- **Deuda neta**: 3.2M euros (reducción significativa desde 8.1M en 2023)

El revenue se distribuye por segmentos:
- **Software**: 32.2M euros (68% del total) con crecimiento del 24%
- **Servicios profesionales**: 15.1M euros (32% del total)

Y por geografía:
- **España**: 21.3M euros (45%)
- **Europa central**: 16.6M euros (35%)
- **Latinoamérica**: 9.5M euros (20%

{'answer': 'Según la información de la vault, **el revenue de la empresa Alpha en 2024 fue de 47.3 millones de euros**, lo que representa un crecimiento del 18.4% comparado con 2023.\n\nDetalles adicionales sobre sus resultados financieros en 2024:\n- **EBITDA**: 12.1M euros (margen del 25.6%)\n- **Beneficio neto**: 6.8M euros\n- **Deuda neta**: 3.2M euros (reducción significativa desde 8.1M en 2023)\n\nEl revenue se distribuye por segmentos:\n- **Software**: 32.2M euros (68% del total) con crecimiento del 24%\n- **Servicios profesionales**: 15.1M euros (32% del total)\n\nY por geografía:\n- **España**: 21.3M euros (45%)\n- **Europa central**: 16.6M euros (35%)\n- **Latinoamérica**: 9.5M euros (20%)',
 'tool_calls': 2,
 'latency': 8.698896646499634}

In [17]:
# Prueba 2: pregunta que requiere navegar wikilinks entre documentos
# El agente debería leer comparativa-financiera y luego seguir
# los links a cada empresa para verificar los datos
vault_agent_query("¿Qué empresa tiene mejor equilibrio entre crecimiento y rentabilidad?")

Pregunta: ¿Qué empresa tiene mejor equilibrio entre crecimiento y rentabilidad?
--------------------------------------------------


/tmp/ipykernel_42099/506747412.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  [1] read_document({'filename': '00-index.md'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma', 'comparativa-financiera', 'riesgos-sector']
  [2] follow_wikilink({'link_name': 'comparativa-financiera'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma']
  [3] follow_wikilink({'link_name': 'empresa-alpha'})
       → wikilinks encontrados: ['comparativa-financiera', 'riesgos-sector']
  [4] follow_wikilink({'link_name': 'empresa-beta'})
       → wikilinks encontrados: ['comparativa-financiera', 'riesgos-sector']
  [5] follow_wikilink({'link_name': 'empresa-gamma'})
       → wikilinks encontrados: ['comparativa-financiera', 'riesgos-sector']

Respuesta:
Basándome en el análisis completo de los documentos, **Empresa Alpha tiene el mejor equilibrio entre crecimiento y rentabilidad**. Aquí está mi razonamiento:

## Análisis Comparativo: Crecimiento vs Rentabilidad

### **Empresa Alpha - Mejor equilibrio**
- **Crecimiento sóli

{'answer': 'Basándome en el análisis completo de los documentos, **Empresa Alpha tiene el mejor equilibrio entre crecimiento y rentabilidad**. Aquí está mi razonamiento:\n\n## Análisis Comparativo: Crecimiento vs Rentabilidad\n\n### **Empresa Alpha - Mejor equilibrio**\n- **Crecimiento sólido**: +18.4% de revenue (47.3M)\n- **Alta rentabilidad**: 25.6% EBITDA margin, 6.8M beneficio neto\n- **Posición financiera saneada**: Deuda neta reducida a 3.2M\n- **Perspectivas positivas**: Objetivo 15-20% crecimiento en 2025 con margen >27%\n\n### **Empresa Beta - Rentabilidad en declive**\n- **Crecimiento bajo**: +6.2% (124.7M revenue)\n- **Margen bajo y cayendo**: 14.7% EBITDA, beneficio neto bajó de 5.4M a 2.1M\n- **Riesgo financiero**: Deuda neta alta (31.4M) y creciendo\n- **Plan de reestructuración**: Reducción de plantilla del 8%\n\n### **Empresa Gamma - Alto crecimiento, sin rentabilidad**\n- **Crecimiento excepcional**: +67% revenue\n- **Sin rentabilidad**: EBITDA y beneficio netos negat

In [18]:
# Prueba 3: pregunta sobre riesgos — tiene que cruzar riesgos-sector con empresa concreta
vault_agent_query("¿Cuáles son los riesgos específicos de invertir en Beta?")

Pregunta: ¿Cuáles son los riesgos específicos de invertir en Beta?
--------------------------------------------------


/tmp/ipykernel_42099/506747412.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  [1] read_document({'filename': '00-index.md'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma', 'comparativa-financiera', 'riesgos-sector']
  [2] follow_wikilink({'link_name': 'empresa-beta'})
       → wikilinks encontrados: ['comparativa-financiera', 'riesgos-sector']
  [3] follow_wikilink({'link_name': 'riesgos-sector'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma']
  [4] follow_wikilink({'link_name': 'comparativa-financiera'})
       → wikilinks encontrados: ['empresa-alpha', 'empresa-beta', 'empresa-gamma']

Respuesta:
Basándome en la información de la vault, estos son los **riesgos específicos de invertir en Beta**:

## Riesgos Principales de Invertir en Beta

### 1. **Riesgo de Márgenes y Rentabilidad**
- Caída dramática del beneficio neto: de 5.4M€ en 2023 a solo 2.1M€ en 2024
- Margen EBITDA bajo del 14.7%, significativamente inferior al 25.6% de Alpha
- Presión continua en márgenes por inflación de costes

{'answer': 'Basándome en la información de la vault, estos son los **riesgos específicos de invertir en Beta**:\n\n## Riesgos Principales de Invertir en Beta\n\n### 1. **Riesgo de Márgenes y Rentabilidad**\n- Caída dramática del beneficio neto: de 5.4M€ en 2023 a solo 2.1M€ en 2024\n- Margen EBITDA bajo del 14.7%, significativamente inferior al 25.6% de Alpha\n- Presión continua en márgenes por inflación de costes\n\n### 2. **Riesgo Energético**\n- Exposición directa a costes energéticos en manufactura industrial\n- Entorno desafiante por inflación de costes energéticos para 2025\n- Vulnerabilidad a volatilidad en precios de energía\n\n### 3. **Riesgo de Concentración Geográfica**\n- Concentración geográfica elevada: 89% del negocio en mercado doméstico (España/Portugal)\n- Solo 11% de exportación (Francia, Italia)\n- Falta de diversificación geográfica aumenta riesgo sistémico\n\n### 4. **Riesgo Financiero**\n- Deuda neta de 31.4M€ y en crecimiento\n- Posición financiera deteriorada c

In [19]:
# Comparativa final: agente con corpus plano vs agente con vault y wikilinks
# en la pregunta que más le costaba antes

pregunta_dificil = "¿Qué empresa tiene mejor ratio LTV/CAC y cuál es su margen EBITDA?"

print("CON CORPUS PLANO (semana 3):")
r_plano = agent_query(pregunta_dificil, verbose=False)
print(f"  Tool calls: {r_plano['tool_calls']} — Latencia: {r_plano['latency']:.2f}s")
print(f"  Respuesta: {r_plano['answer'][:200]}")

print("\nCON VAULT Y WIKILINKS:")
r_vault = vault_agent_query(pregunta_dificil, verbose=False)
print(f"  Tool calls: {r_vault['tool_calls']} — Latencia: {r_vault['latency']:.2f}s")
print(f"  Respuesta: {r_vault['answer'][:200]}")

print("\nObserva: ¿el agente con wikilinks necesita menos tool calls para llegar a la misma respuesta?")
print("Eso es la ventaja del grafo: la estructura del conocimiento guía la navegación.")

CON CORPUS PLANO (semana 3):


/tmp/ipykernel_42099/1195034023.py:11: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = client_anthropic.messages.create(


  Tool calls: 11 — Latencia: 20.53s
  Respuesta: Perfecto, ahora tengo toda la información necesaria. Permíteme resumir los hallazgos:

## Ratio LTV/CAC

**Empresa Gamma** es la única que proporciona datos específicos sobre LTV/CAC:
- **CAC**: 18.40

CON VAULT Y WIKILINKS:
  Tool calls: 5 — Latencia: 21.42s
  Respuesta: ¡Perfecto! Encontré la información completa. Solo Empresa Gamma (la empresa SaaS) tiene reportadas las métricas LTV/CAC, lo cual tiene sentido ya que estas métricas son específicas de modelos de negoc

Observa: ¿el agente con wikilinks necesita menos tool calls para llegar a la misma respuesta?
Eso es la ventaja del grafo: la estructura del conocimiento guía la navegación.


<div style="font-size:14px">

### La conclusión práctica

Los **wikilinks** no reducen la latencia de forma dramática en datasets pequeños.  
Su ventaja real aparece cuando la **vault crece**.

Con 200 documentos, el agente con corpus plano haría **40-50 tool calls** explorando a ciegas.

El agente con wikilinks seguiría haciendo **5-6**, porque la estructura le guía igual.

Ahí es donde la diferencia de latencia se vuelve **brutal**.

</div>